### Variables

In [937]:
import os
import pandas as pd
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics.pairwise import cosine_distances

RESUME_ID = "72b39379-e2da-4b28-9102-a32b77eacd97"
SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.66
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.66
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
SOFT_SKILLS_STRING_COLUMN_INDEX = 4
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3

LLM_MODEL_VECTOR_DIMENSIONS = 3072
DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings_general"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

### Functions Definitions

In [938]:
def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=db_user, password=db_pw
    )
    register_vector(conn)
    return conn

def get_resume(resume_id: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s", (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    filter_string = f"WHERE ai_industries::TEXT[] && ARRAY[{industries}]"
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} {filter_string}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)", (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_candidate_skills(resume_id: str, table_name: str) -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_name} WHERE resume_id = '{resume_id}'")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)    

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  

def get_compliance_mask(
        compliance_type: str,
        cosine_similarities: np.ndarray, 
        threshold: float,
        weight_matrix: np.ndarray = None, 
        weight: float = 0):    
    if compliance_type == "COMPLIANT":
        binary_mask = (cosine_similarities > threshold).astype(np.int8)
    elif compliance_type == "NONCOMPLIANT":
        binary_mask = (cosine_similarities <= threshold).astype(np.int8)    
    elif compliance_type == "IDEAL" and weight != 0 and weight_matrix is not None:
        minimum_weights_matrix = np.where(binary_mask == 1, weight_matrix, 0)    
        binary_mask = ((weight >= minimum_weights_matrix) & (binary_mask == 1)).astype(np.int8)
    return binary_mask

def create_mapping_matrix(count_list: list):
    nrows = len(count_list)
    ncols = max(count_list)
    matrix = np.zeros((nrows, ncols), dtype=np.int8)
    for i, count in enumerate(count_list):
        matrix[i, :count] = 1
    return matrix     

def analyze_market(market_obj: MarketSkillsMatrix, candidate_skills_df: pd.DataFrame, skills_type: str) -> None:
    weight_column_index = SOFT_SKILLS_WEIGHT_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_WEIGHT_COLUMN_INDEX
    string_column_index = SOFT_SKILLS_STRING_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_STRING_COLUMN_INDEX
    threshold = SOFT_SKILLS_SIMILARITY_THRESHOLD if skills_type == "soft" else HARD_SKILLS_SIMILARITY_THRESHOLD
    skills_count = candidate_skills_df.shape[0]

    for i in range(0, skills_count):
        weight = candidate_skills_df.iloc[(i, weight_column_index)] 
        skill_embedding = candidate_skills_df.iloc[(i, string_column_index) ] 
        cosine_similarities = cosine_similarities_matrix(skill_embedding, market_obj.embedding_matrix)
        binary_mask = get_compliance_mask("COMPLIANT", cosine_similarities, threshold)        
        market_obj.accumulate_matches(binary_mask)
        market_obj.accumulate_weighted_matches(weight, binary_mask)

class MarketSkillsMatrix:
    _MATCH_SCORE_THRESHOLD = 1

    def __init__(self, skill_type: str, jobs_df: pd.DataFrame):
        if skill_type not in ("hard", "soft"):
            raise ValueError(f"skill_type must be 'hard' or 'soft', got '{skill_type}'")
        if jobs_df.empty:
            raise ValueError("jobs_df cannot be empty")

        self.skill_type = skill_type

        # Populated by _initialize_matrices
        self.string_matrix: np.ndarray = None      # (n_jobs, max_skills) skill descriptions
        self.weight_matrix: np.ndarray = None      # (n_jobs, max_skills) skill weights
        self.embedding_matrix: np.ndarray = None   # (n_jobs, max_skills, vector_dim) embeddings
        self.count_by_index: list[int] = []        # count of skills per job
        self.job_id_by_index: list = []            # job_id mapped to matrix row index

        # Populated after combine() / weight_against() calls
        self._match_score_matrix: np.ndarray = None     # raw match counts per (job, skill) cell
        self._weighted_match_matrix: np.ndarray = None  # weight-qualified match counts

        self._initialize_matrices(jobs_df)

    def _initialize_matrices(self, jobs_df: pd.DataFrame) -> None:
        matching_jobs_ids = jobs_df["id"].tolist()
        table = SOFT_SKILLS_TABLE if self.skill_type == "soft" else HARD_SKILLS_TABLE
        skills_df = get_position_skills(matching_jobs_ids, table).sort_values("job_id")

        skills_df["index"] = skills_df.groupby("job_id").ngroup()
        max_skills_per_job = skills_df.groupby("index").size().max()

        self.string_matrix = self._build_padded_matrix(
            skills_df, "skill_description", max_skills_per_job, pad_value=""
        )
        self.weight_matrix = self._build_padded_matrix(
            skills_df, "weight", max_skills_per_job, pad_value=0, dtype=np.float32
        )
        self.embedding_matrix = self._build_embedding_matrix(
            skills_df, max_skills_per_job
        )
        self.count_by_index = (
            skills_df["index"].value_counts().sort_index().tolist()
        )
        self.job_id_by_index = (
            skills_df[["index", "job_id"]].drop_duplicates()["job_id"].tolist()
        )
        self._match_score_matrix = create_mapping_matrix(self.count_by_index)
        self._weighted_match_matrix = np.zeros_like(self._match_score_matrix)

    def _build_padded_matrix(
        self,
        skills_df: pd.DataFrame,
        column: str,
        max_skills: int,
        pad_value,
        dtype=None,
    ) -> np.ndarray:
        rows = [
            np.pad(
                group[column].values,
                (0, max_skills - len(group)),
                constant_values=pad_value,
            )
            for _, group in skills_df.groupby("index")
        ]
        return np.array(rows, dtype=dtype)

    def _build_embedding_matrix(
        self, skills_df: pd.DataFrame, max_skills: int
    ) -> np.ndarray:
        zero_vector = np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)
        rows = [
            np.vstack(
                list(group["embedding"].values)
                + [zero_vector] * (max_skills - len(group))
            )
            for _, group in skills_df.groupby("index")
        ]
        return np.array(rows, dtype=np.float32)

    def accumulate_matches(self, match_array: np.ndarray) -> None:
        """Add a match score array into the running match score matrix."""
        if match_array.shape != self._match_score_matrix.shape:
            raise ValueError(
                f"match_array shape {match_array.shape} does not match "
                f"expected {self._match_score_matrix.shape}"
            )
        self._match_score_matrix += match_array

    def accumulate_weighted_matches(
        self, skill_weight: float, binary_mask: np.ndarray
    ) -> None:
        """Record which job skills are met by a candidate skill at the given weight."""
        if binary_mask.shape != self.weight_matrix.shape:
            raise ValueError(
                f"binary_mask shape {binary_mask.shape} does not match "
                f"weight_matrix shape {self.weight_matrix.shape}"
            )
        candidate_weight_mask = binary_mask * skill_weight
        weight_qualified = (candidate_weight_mask >= self.weight_matrix) & (self.weight_matrix != 0)
        self._weighted_match_matrix += weight_qualified.astype(np.int8)

    def get_minimum_compliance_by_job(self) -> list[float]:
        """
        Percentage of a job's skills matched above the minimum score threshold,
        per job index. A cell qualifies if its match score exceeds _MATCH_SCORE_THRESHOLD.
        """
        qualifying = (self._match_score_matrix > self._MATCH_SCORE_THRESHOLD).sum(axis=1)
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.count_by_index)]

    def get_ideal_compliance_by_job(self) -> list[float]:
        """
        Percentage of a job's skills met at or above their required weight,
        per job index.
        """
        qualifying = (self._weighted_match_matrix != 0).sum(axis=1)
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.count_by_index)]

### Get Candidate Info

In [939]:
RESUME_ID = "ed8a492e-d72b-488d-924a-e198c027aa79"  # REMOVE  
resume_df = get_resume(RESUME_ID)
candidate_industries = resume_df["industries"][0]

candidate_hard_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_HARD_SKILLS_TABLE)
candidate_soft_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_SOFT_SKILLS_TABLE)

jobs_df = filter_job_postings(candidate_industries)

soft_market = MarketSkillsMatrix("soft", jobs_df.copy())
hard_market = MarketSkillsMatrix("hard", jobs_df.copy())

Find candidate's best matches for each skill  
Consider soft and hard skills

In [940]:
def get_market_analysis_results(market_obj: MarketSkillsMatrix) -> list[dict]:
    compliance_by_job = market_obj.get_minimum_compliance_by_job()
    ideal_compliance_by_job = market_obj.get_ideal_compliance_by_job()
    noncompliance_mask = market_obj._match_score_matrix == 1

    return [
        {
            "job_id": job_id,
            "job_index": i,
            "minimum_compliance_pct": compliance_pct,
            "ideal_compliance_pct": ideal_compliance_pct,
            "nonmatched_skills_count": int(noncompliance_mask[i].sum()),
            "nonmatched_skills": list(set(market_obj.string_matrix[i][noncompliance_mask[i]])),
            "similarity_matched_skills": _get_similarity_matched_skills(market_obj, i),
            "similarity_match_scores": _get_similarity_matched_skills(market_obj, i, with_scores=True),
            "not_ideal_skills": list(set(
                    market_obj.string_matrix[i][
                        (market_obj._weighted_match_matrix[i] == 0) &
                        (market_obj.string_matrix[i] != "")
                    ]
                ))
        }
        for i, (job_id, compliance_pct, ideal_compliance_pct) in enumerate(
            zip(market_obj.job_id_by_index, compliance_by_job, ideal_compliance_by_job)
        )
    ]

def _get_similarity_matched_skills(market_obj: MarketSkillsMatrix, job_index: int, top_n: int = 5, with_scores: bool = False):
    scores = market_obj._match_score_matrix[job_index]
    descriptions = market_obj.string_matrix[job_index]
    ranked = sorted(
        ((desc, int(score)) for desc, score in zip(descriptions, scores) if desc != "" and score > 0),
        key=lambda x: x[1],
        reverse=True,
    )
    ranked = ranked[:top_n]
    return ranked if with_scores else [desc for desc, _ in ranked]

def print_analysis(market_obj: MarketSkillsMatrix, analysis: list[dict]) -> None:
    sorted_analysis = sorted(analysis, key=lambda e: e["minimum_compliance_pct"], reverse=True)
    sorted_counts = [market_obj.count_by_index[market_obj.job_id_by_index.index(e["job_id"])] for e in sorted_analysis]

    print("Count of required skills by job:", sorted_counts)
    print()
    for entry in sorted_analysis:
        print(
            f"[{entry['job_id']} | {entry['job_index']}]\n"
            f"  minimum: {entry['minimum_compliance_pct']}% | ideal: {entry['ideal_compliance_pct']}%\n"
            f"  present skills matches ({len(entry['similarity_matched_skills'])}):  {entry['similarity_matched_skills']}\n"
            f"  insufficient proficiency ({len(entry['not_ideal_skills'])}): {entry['not_ideal_skills']}\n"
            f"  nonmatched ({entry['nonmatched_skills_count']}): {entry['nonmatched_skills']}"
        )

def print_popular_similarity_matches(analysis: list[dict]) -> None:
    skill_counts: dict[str, int] = {}
    for entry in analysis:
        for skill, score in entry["similarity_match_scores"]:
            skill_counts[skill] = skill_counts.get(skill, 0) + score

    print("Most popular similarity matches (across all jobs):")
    for skill, count in sorted(skill_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {skill}: {count}")

In [941]:
analyze_market(soft_market, candidate_soft_skills_df, "soft")
market_soft_skills_analysis = get_market_analysis_results(soft_market)
print_analysis(soft_market, market_soft_skills_analysis)

Count of required skills by job: [5, 5, 4, 7, 4, 5, 5, 4, 5, 7, 5, 3, 3, 5, 6, 6, 7, 4, 4, 5, 7, 4, 4, 5, 5, 5, 6, 5, 10, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 11, 5, 5, 10, 10, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 9, 9, 4, 4, 8, 8, 8, 4, 8, 4, 8, 4, 4, 4, 4, 4, 11, 7, 7, 7, 14, 7, 7, 14, 7, 7, 7, 7, 7, 21, 7, 7, 10, 10, 10, 16, 12, 6, 6, 9, 6, 6, 6, 6, 6, 9, 6, 3, 12, 6, 12, 3, 6, 9, 9, 6, 6, 6, 9, 14, 14, 14, 11, 8, 8, 8, 8, 8, 8, 21, 13, 13, 10, 10, 10, 10, 10, 10, 10, 10, 10, 15, 5, 5, 5, 10, 5, 5, 5, 10, 5, 12, 12, 12, 7, 7, 14, 14, 7, 7, 16, 9, 9, 9, 9, 9, 9, 18, 9, 20, 11, 11, 11, 11, 11, 11, 11, 22, 13, 13, 13, 8, 4, 12, 10, 16, 10, 8, 8, 6, 12, 8, 4, 10, 4, 4, 4, 6, 8, 6, 6, 8, 12, 14, 13, 24, 11, 11, 11, 11, 9, 9, 9, 9, 9, 9, 9, 7, 7, 7, 7, 7, 7, 7, 7, 14, 12, 12, 12, 17, 17, 15, 10, 15, 15, 15, 5, 20, 5, 15, 15, 5, 10, 5, 10, 18, 21, 16, 8, 8, 8, 8, 19, 11, 11, 11, 14, 17, 17, 20, 9, 18, 6, 9, 9, 12, 6, 22, 16, 16, 13, 10, 10, 10, 10, 34, 14, 14, 18, 8, 33, 17

In [942]:
analyze_market(hard_market, candidate_hard_skills_df, "hard")
market_hard_skills_analysis = get_market_analysis_results(hard_market)
print_analysis(hard_market, market_hard_skills_analysis)

Count of required skills by job: [16, 20, 16, 23, 22, 18, 18, 24, 26, 17, 16, 31, 31, 28, 27, 20, 20, 31, 12, 12, 18, 12, 29, 23, 17, 17, 28, 27, 16, 16, 32, 16, 36, 25, 20, 10, 19, 19, 14, 37, 23, 18, 18, 13, 30, 17, 21, 21, 21, 37, 45, 16, 27, 19, 19, 30, 15, 15, 15, 30, 37, 37, 11, 22, 22, 88, 18, 25, 25, 32, 14, 14, 24, 24, 34, 17, 20, 33, 46, 23, 36, 49, 13, 39, 26, 26, 13, 26, 29, 19, 22, 28, 40, 39, 18, 42, 21, 27, 42, 6, 24, 27, 41, 41, 35, 26, 26, 26, 26, 23, 43, 20, 40, 20, 37, 34, 17, 17, 17, 31, 45, 42, 36, 36, 47, 33, 22, 30, 30, 30, 27, 27, 27, 102, 24, 16, 21, 47, 26, 26, 31, 31, 31, 49, 23, 33, 38, 15, 15, 55, 25, 45, 15, 15, 30, 25, 35, 42, 32, 54, 39, 39, 17, 34, 46, 41, 24, 26, 33, 21, 21, 21, 35, 42, 58, 37, 37, 30, 39, 16, 16, 75, 84, 52, 54, 64, 33, 39, 13, 15, 15, 62, 47, 94, 32, 19, 19, 21, 79, 27, 27, 31, 35, 43, 54, 56, 32, 34, 16, 20, 24, 20, 20, 47, 47, 43, 31, 25, 21, 21, 19, 38, 53, 17, 43, 39, 11, 22, 22, 51, 49, 38, 23, 114, 14, 28, 59, 26, 52, 31, 51, 4

In [943]:
print_popular_similarity_matches(market_soft_skills_analysis)
print_popular_similarity_matches(market_hard_skills_analysis)

Most popular similarity matches (across all jobs):
  Problem Solving: 334
  Collaboration: 298
  Adaptability: 258
  Stakeholder Communication: 231
  Cross-functional Collaboration: 116
  Communication: 112
  Problem-Solving: 86
  Continuous Learning: 78
  Proactive Communication: 60
  Analytical Skills: 60
  Teamwork: 56
  Continuous Improvement: 44
  Mentorship: 36
  Communication Skills: 34
  Learning Agility: 33
  Analytical Thinking: 32
  Written Communication: 30
  Mentoring: 26
  Technical Communication: 22
  Strategic Thinking: 22
  Team Collaboration: 20
  Clear Communication: 18
  Stakeholder Collaboration: 16
  Proactive Problem Solving: 16
  Scalability Focus: 16
  Root Cause Analysis: 14
  Problem-solving: 14
  Stakeholder Management: 14
  Innovation: 12
  Project Management: 12
  Can-do Attitude: 12
  Resilience: 12
  Proactive Learning: 10
  Troubleshooting: 10
  Proactivity: 10
  Business Acumen: 10
  Critical Thinking: 10
  Interpersonal Communication: 10
  Data-Driven

### Test clusterization

In [ ]:
noncompliance_mask = hard_market._match_score_matrix == 1

missing_skills_matrix = np.where(
    noncompliance_mask[:, :, np.newaxis],
    hard_market.embedding_matrix,
    np.nan
)

# Flatten and remove nans
flat = missing_skills_matrix.reshape(-1, LLM_MODEL_VECTOR_DIMENSIONS)
valid_mask = ~np.isnan(flat).any(axis=1)
missing_skills_matrix = flat[valid_mask]

# Find good eps
distances = cosine_distances(missing_skills_matrix)
nearest = np.sort(distances, axis=1)[:, 1]
eps = np.percentile(nearest, 50)
print(f"Using eps: {eps}")

# DBSCAN to find k
labels_dbscan = DBSCAN(eps=eps, min_samples=2, metric='cosine').fit_predict(missing_skills_matrix)
k = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
print(f"DBSCAN found {k} clusters")

# KMeans with k
kmeans = KMeans(n_clusters=k)
labels_kmeans = kmeans.fit_predict(missing_skills_matrix)

# Group vectors
grouped = {}
outlier_count = 0
for embedding, dbscan_label, kmeans_label in zip(missing_skills_matrix, labels_dbscan, labels_kmeans):
    if dbscan_label == -1:
        grouped[f"outlier_{outlier_count}"] = [embedding]
        outlier_count += 1
    else:
        grouped.setdefault(kmeans_label, []).append(embedding)

grouped = list(grouped.values())
print(f"Total groups: {len(grouped)}")